[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nia-04/CSPC-513-ML-Lab/blob/main/Practical_05_Logistic_Regression.ipynb)

# Practical 05: Implementation of Logistic Regression

## Aim

To implement Logistic Regression using Python and Scikit-learn to predict whether a student passes or fails based on their subject marks.

## Introduction

Logistic Regression is a supervised machine learning algorithm used for classification problems. It predicts the probability that an input belongs to a particular class.

In this practical, Logistic Regression is applied to a preprocessed student dataset to predict the student's result based on marks in five subjects.

## Theory

Logistic Regression uses the sigmoid function to convert a linear combination of input features into a probability between 0 and 1.

The sigmoid function is:

$$
P(y=1)=\frac{1}{1+e^{-z}}
$$

where:

$$
z=b_0+b_1x_1+b_2x_2+\cdots+b_nx_n
$$

Here:

* \(x_1,x_2,\ldots,x_n\) represent the input features.
* \(b_0\) is the intercept.
* \(b_1,b_2,\ldots,b_n\) are the model coefficients.
* \(P(y=1)\) is the probability of belonging to class 1.

A probability of 0.5 or greater is generally classified as class 1; otherwise, it is classified as class 0.


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Libraries imported successfully.")

Libraries imported successfully.


In [19]:
import os
import pandas as pd

if os.path.exists("EXPORTS/Mldata_Preprocessed.xlsx"):
    dataset_path = "EXPORTS/Mldata_Preprocessed.xlsx"
else:
    !git clone https://github.com/Nia-04/CSPC-513-ML-Lab.git
    %cd CSPC-513-ML-Lab
    dataset_path = "EXPORTS/Mldata_Preprocessed.xlsx"

df = pd.read_excel(dataset_path)

df.head()

,ROLL NO.,GENDER,SOFTWARE ENGINEER,TOC,COMPUTER NETWORK,MACHINE LEARNING,DATA WAREHOUSE,GRADE,RESULT
0,240104010001,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1
1,240104010002,F,0.500000,0.661017,0.741379,0.649123,0.545455,B,1
2,240104010003,M,0.633333,0.627119,0.620690,0.614035,0.600000,B,1
3,240104010004,M,0.983333,0.915254,0.948276,0.964912,0.890909,O,1
4,240104010005,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1


## Step 1: Load the Dataset

Load the preprocessed dataset `Mldata_preprocessed.xlsx` into a Pandas DataFrame.

The dataset contains student roll numbers, gender, normalized subject marks, grades, and results.

Since the data has already been preprocessed, we do not need to repeat the preprocessing steps.


In [11]:
# Load the preprocessed dataset
df = pd.read_excel("EXPORTS/Mldata_preprocessed.xlsx")

# Display the first five rows
df.head()

,ROLL NO.,GENDER,SOFTWARE ENGINEER,TOC,COMPUTER NETWORK,MACHINE LEARNING,DATA WAREHOUSE,GRADE,RESULT
0,240104010001,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1
1,240104010002,F,0.500000,0.661017,0.741379,0.649123,0.545455,B,1
2,240104010003,M,0.633333,0.627119,0.620690,0.614035,0.600000,B,1
3,240104010004,M,0.983333,0.915254,0.948276,0.964912,0.890909,O,1
4,240104010005,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1


## Step 2: Select Input Features and Target Variable

The five subject marks are selected as input features (X):

* Software Engineering
* TOC
* Computer Network
* Machine Learning
* Data Warehouse

The `RESULT` column is selected as the target variable (y).

Here, 1 represents Pass and 0 represents Fail.

The columns `ROLL NO.`, `GENDER`, and `GRADE` are excluded from the input features to avoid unnecessary information and target leakage.


In [12]:
# Select input features
X = df[
    [
        'SOFTWARE ENGINEER',
        'TOC',
        'COMPUTER NETWORK',
        'MACHINE LEARNING',
        'DATA WAREHOUSE'
    ]
]

# Select target variable
y = df['RESULT']

print("Input features shape:", X.shape)
print("Target variable shape:", y.shape)

print("\nTarget class distribution:")
print(y.value_counts())

Input features shape: (67, 5)
Target variable shape: (67,)

Target class distribution:
RESULT
1    64
0     3
Name: count, dtype: int64


## Step 3: Split the Dataset

The dataset is divided into training and testing sets.

* **Training set:** 80% of the data, used to train the model.
* **Testing set:** 20% of the data, used to evaluate the model.

The `random_state=42` parameter ensures reproducible results. Stratification maintains approximately the same proportion of Pass and Fail examples in both sets.


In [13]:
# Check the population of each class
print("Population of each class:")
print(y.value_counts().sort_index())

print("\nClass distribution (%):")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))

# Split the dataset while preserving class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set population:")
print(y_train.value_counts().sort_index())

print("\nTesting set population:")
print(y_test.value_counts().sort_index())

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

Population of each class:
RESULT
0     3
1    64
Name: count, dtype: int64

Class distribution (%):
RESULT
0     4.48
1    95.52
Name: proportion, dtype: float64

Training set population:
RESULT
0     2
1    51
Name: count, dtype: int64

Testing set population:
RESULT
0     1
1    13
Name: count, dtype: int64

Training samples: 53
Testing samples: 14


## Step 4: Create and Train the Logistic Regression Model

An instance of `LogisticRegression` is created and trained using the training dataset.

The model learns the relationship between the subject marks and the student's result.


In [14]:
# Create the Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## Step 5: Predict the Results

The trained model is used to predict the results of students in the testing dataset.

The predicted values are compared with the actual results to evaluate model performance.


In [15]:
# Predict test results
y_pred = model.predict(X_test)

# Display actual and predicted results
results = pd.DataFrame({
    'Actual Result': y_test.values,
    'Predicted Result': y_pred
})

print(results.to_string(index=False))

 Actual Result  Predicted Result
             1                 1
             1                 1
             1                 1
             1                 1
             1                 1
             0                 1
             1                 1
             1                 1
             1                 1
             1                 1
             1                 1
             1                 1
             1                 1
             1                 1


## Step 6: Evaluate Model Performance

The model is evaluated using the following metrics:

1. **Accuracy:** Percentage of correctly classified samples.
2. **Confusion Matrix:** Shows correct and incorrect predictions for each class.
3. **Classification Report:** Displays precision, recall, and F1-score for each class.


In [16]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model Accuracy: 92.86 %

Confusion Matrix:
[[ 0  1]
 [ 0 13]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.93      1.00      0.96        13

    accuracy                           0.93        14
   macro avg       0.46      0.50      0.48        14
weighted avg       0.86      0.93      0.89        14



## Step 7: Predict the Result of a New Student

The user enters normalized marks for the five subjects.

The trained Logistic Regression model predicts whether the student is likely to Pass or Fail and displays the predicted probability of passing.

**Note:** Each input mark must be between 0 and 1 because the dataset contains normalized marks.


In [18]:
print("Enter the student's actual marks.")
print("Enter marks out of the maximum marks for each subject.\n")

# Get actual marks from the user
se = float(input("Software Engineering: "))
toc = float(input("TOC: "))
cn = float(input("Computer Network: "))
ml = float(input("Machine Learning: "))
dw = float(input("Data Warehouse: "))

actual_marks = [se, toc, cn, ml, dw]

# Maximum marks for each subject in the original dataset
max_marks = [60, 59, 58, 57, 55]

# Validate the marks
if any(mark < 0 or mark > maximum
       for mark, maximum in zip(actual_marks, max_marks)):

    print("\nError: Enter marks within the valid range for each subject.")

else:
    # Normalize the marks using the same scale as the dataset
    normalized_marks = [
        mark / maximum
        for mark, maximum in zip(actual_marks, max_marks)
    ]

    # Create input DataFrame with the correct feature names
    new_student = pd.DataFrame(
        [normalized_marks],
        columns=X.columns
    )

    # Display normalized marks
    print("\nNormalized Marks:")
    print(new_student.round(4).to_string(index=False))

    # Predict result
    prediction = model.predict(new_student)[0]

    # Calculate probability of passing
    pass_probability = model.predict_proba(new_student)[0][
        list(model.classes_).index(1)
    ]

    print("\nPredicted Result:",
          "PASS" if prediction == 1 else "FAIL")

    print("Probability of Passing:",
          round(pass_probability * 100, 2), "%")

Enter the student's actual marks.
Enter marks out of the maximum marks for each subject.


Normalized Marks:
 SOFTWARE ENGINEER    TOC  COMPUTER NETWORK  MACHINE LEARNING  DATA WAREHOUSE
            0.5667 0.7627            0.3793            0.2456          0.9273

Predicted Result: PASS
Probability of Passing: 95.27 %


## Result

Logistic Regression was successfully implemented using Python and Scikit-learn on the preprocessed student dataset. The model was trained to predict student results based on marks in five subjects and evaluated using accuracy, a confusion matrix, and a classification report. The trained model was also used to predict the result of a new student based on user-provided input.
